# YouTube AI-Slop: transcript-routed synthetic-voice experiment

This Colab notebook tests whether an AI-writing detector can **route** expensive synthetic-voice inference toward useful parts of a video.

It does not treat AI-written text as proof of AI voice. The complete timestamped YouTube transcript is scored at several overlapping scales; every detected text region is then sliced into 4-second audio clips for the voice model.

> **Runtime:** select **Runtime → Change runtime type → T4 GPU** before running.  
> **License:** the Arena-hosted audio detector is CC BY-NC-SA 4.0. Use this notebook for evaluation/prototyping only unless you separately resolve model licensing.

## Goal

Produce inspectable, clip-level evidence:

- timestamped transcript;
- AI-writing scores over the complete transcript at 24-, 48-, and 96-word scales;
- merged timestamp regions whose text score passes the routing threshold;
- 4-second synthetic-voice scores for every slice inside those regions; and
- conservative video-level aggregates for comparing experiments.

The output is not a production block decision.

## Setup

Install the Colab dependencies. PyTorch and FFmpeg are provided by Colab.

In [ ]:
from pathlib import Path
import subprocess
import sys

REQUIREMENTS_URL = "https://raw.githubusercontent.com/emilzmmn04/youtube-ai-slop-audio-detector/main/requirements-colab.txt"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", REQUIREMENTS_URL])
print("Dependencies installed. If Colab asks for a runtime restart, restart and continue here.")

## Configuration

Start with the defaults. Lowering the text threshold increases recall and sends more timestamped audio to the voice model.

In [ ]:
# A YouTube URL is required for captions. If audio download is blocked, upload the same media when prompted.
YOUTUBE_URL = ""  # @param {type:"string"}

# Models
TEXT_MODEL_ID = "GeorgeDrayson/modernbert-ai-detection-raid-mage"
AUDIO_MODEL_ID = "SpeechAntiSpoofingBenchmarks/Wav2Vec2-Small-AntiDeepfake-NDA"

# Full-transcript, multi-scale routing
TEXT_WINDOW_SIZES = (24, 48, 96)
TEXT_STRIDE_DIVISOR = 3
TEXT_ROUTE_THRESHOLD = 0.50  # @param {type:"number"}
TEXT_REGION_PADDING_SECONDS = 1.0  # @param {type:"number"}

# The Arena ONNX export uses exactly 64,000 samples at 16 kHz (4 seconds).
SAMPLE_RATE = 16_000
AUDIO_CLIP_SAMPLES = 64_000
AUDIO_CLIP_SECONDS = AUDIO_CLIP_SAMPLES / SAMPLE_RATE
AUDIO_STRIDE_SECONDS = 2.0  # @param {type:"number"}
AUDIO_BATCH_SIZE = 16  # @param {type:"integer"}
PROVISIONAL_AUDIO_THRESHOLD = 0.80  # @param {type:"number"}

assert all(8 <= size <= 500 for size in TEXT_WINDOW_SIZES)
assert 0.0 <= TEXT_ROUTE_THRESHOLD <= 1.0
assert 0 < AUDIO_STRIDE_SECONDS <= AUDIO_CLIP_SECONDS
print(f"Audio clips: {AUDIO_CLIP_SECONDS:.4f} seconds each")

## Step 1 — Acquire and normalize media

The notebook first tries `yt-dlp`. If YouTube blocks Colab's datacenter IP, it opens a file uploader while preserving the URL for caption retrieval. All input is converted to 16 kHz mono WAV.

In [ ]:
from pathlib import Path

WORKDIR = Path("/content/ai_slop_detector")
WORKDIR.mkdir(parents=True, exist_ok=True)
SOURCE_PATH = WORKDIR / "input_media"
AUDIO_PATH = WORKDIR / "input_audio.wav"

# Colab IPs are often challenged by YouTube. Install yt-dlp's recommended PO-token provider remotely.
pot_repo = WORKDIR / "bgutil-ytdlp-pot-provider"
pot_server = pot_repo / "server"
if not (pot_server / "build" / "main.js").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "1.3.1",
                    "https://github.com/Brainicism/bgutil-ytdlp-pot-provider.git", str(pot_repo)], check=True)
    subprocess.run(["npm", "ci"], cwd=pot_server, check=True)
    subprocess.run(["npx", "tsc"], cwd=pot_server, check=True)

source_media = None
if YOUTUBE_URL.strip():
    output_template = str(WORKDIR / "download.%(ext)s")
    command = [
        "yt-dlp", "--no-playlist", "-f", "bestaudio/best",
        "--js-runtimes", "node",
        "--extractor-args", "youtube:player_client=web_safari",
        "--extractor-args", f"youtubepot-bgutilscript:server_home={pot_server}",
        "-o", output_template, YOUTUBE_URL.strip(),
    ]
    try:
        subprocess.run(command, check=True)
        downloaded = [p for p in WORKDIR.glob("download.*") if p.is_file()]
        if downloaded:
            source_media = downloaded[0]
    except subprocess.CalledProcessError:
        print("YouTube blocked the Colab downloader; upload the same video's audio/video below.")

if source_media is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("File upload requires Colab; alternatively set YOUTUBE_URL") from exc
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded")
    uploaded_name = next(iter(uploaded))
    source_media = WORKDIR / Path(uploaded_name).name
    source_media.write_bytes(uploaded[uploaded_name])

subprocess.run([
    "ffmpeg", "-y", "-i", str(source_media), "-vn",
    "-ac", "1", "-ar", str(SAMPLE_RATE), "-c:a", "pcm_s16le", str(AUDIO_PATH),
], check=True, capture_output=True)

print(f"Normalized audio: {AUDIO_PATH}")
print(f"Size: {AUDIO_PATH.stat().st_size / 1_000_000:.1f} MB")

## Step 2 — Fetch YouTube captions with timestamps

The notebook uses the video's existing YouTube caption track instead of running ASR. Caption-segment times are interpolated across words so text windows can be mapped back to audio.

In [ ]:
import html
import json
import re
from urllib.parse import parse_qs, urlparse
from youtube_transcript_api import YouTubeTranscriptApi


def youtube_video_id(url):
    parsed = urlparse(url.strip())
    if parsed.netloc in {"youtu.be", "www.youtu.be"}:
        return parsed.path.strip("/").split("/")[0]
    if parsed.netloc.endswith("youtube.com"):
        if parsed.path == "/watch":
            return parse_qs(parsed.query).get("v", [None])[0]
        parts = parsed.path.strip("/").split("/")
        if len(parts) >= 2 and parts[0] in {"shorts", "embed", "live"}:
            return parts[1]
    return None


VIDEO_ID = youtube_video_id(YOUTUBE_URL)
if not VIDEO_ID:
    raise ValueError("Set YOUTUBE_URL to a valid YouTube video so captions can be fetched.")

api = YouTubeTranscriptApi()
tracks = list(api.list(VIDEO_ID))
if not tracks:
    raise RuntimeError("This video exposes no YouTube caption tracks.")
tracks.sort(key=lambda track: (
    not str(track.language_code).lower().startswith("en"),
    bool(track.is_generated),
))
track = tracks[0]
captions = track.fetch()

words = []
full_text = []
for item in captions:
    clean_text = html.unescape(re.sub(r"<[^>]+>", " ", item.text)).replace("\n", " ")
    tokens = clean_text.split()
    if not tokens:
        continue
    start_s = float(item.start)
    duration_s = max(float(item.duration), 0.01)
    full_text.extend(tokens)
    for index, token in enumerate(tokens):
        words.append({
            "text": token,
            "start_s": start_s + duration_s * index / len(tokens),
            "end_s": start_s + duration_s * (index + 1) / len(tokens),
        })

if not words:
    raise RuntimeError("The selected YouTube caption track was empty.")

transcript_record = {
    "video_id": VIDEO_ID,
    "source": "youtube_captions",
    "language": track.language_code,
    "is_generated": bool(track.is_generated),
    "text": " ".join(full_text),
    "words": words,
}
(WORKDIR / "transcript.json").write_text(json.dumps(transcript_record, indent=2), encoding="utf-8")
print(
    f"YouTube captions: {len(words):,} timestamped words, "
    f"{words[-1]['end_s'] / 60:.1f} min, "
    f"language={track.language_code}, auto_generated={track.is_generated}"
)
print(transcript_record["text"][:800])

## Step 3 — Build and score transcript windows

The complete YouTube transcript is scored in overlapping 24-, 48-, and 96-word windows. Short windows localize narration; longer windows add context. The score is a routing signal, not proof that the text or voice is synthetic.

In [ ]:
import gc
import time
from pathlib import Path
from urllib.parse import urlparse
import numpy as np
import pandas as pd
import requests
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer


def build_text_windows(word_records, window_sizes, stride_divisor):
    rows = []
    total = len(word_records)
    for window_words in window_sizes:
        stride_words = max(1, window_words // stride_divisor)
        starts = list(range(0, max(total - window_words + 1, 1), stride_words))
        final_start = max(total - window_words, 0)
        if not starts or starts[-1] != final_start:
            starts.append(final_start)
        for start_index in sorted(set(starts)):
            end_index = min(start_index + window_words, total)
            window = word_records[start_index:end_index]
            rows.append({
                "window_size": window_words,
                "start_word": start_index,
                "end_word": end_index,
                "start_s": window[0]["start_s"],
                "end_s": window[-1]["end_s"],
                "word_count": len(window),
                "text": " ".join(item["text"] for item in window),
            })
    for window_id, row in enumerate(rows):
        row["window_id"] = window_id
    return pd.DataFrame(rows)


def download_hf_large_file(repo_id, filename, destination, attempts=20, chunk_size=8 * 1024 * 1024):
    destination = Path(destination)
    signed_url = None
    for attempt in range(attempts):
        resolve_url = (
            f"https://huggingface.co/{repo_id}/resolve/main/{filename}"
            f"?download=true&fresh={time.time_ns()}"
        )
        redirect = requests.get(resolve_url, allow_redirects=False, headers={"Cache-Control": "no-cache"}, timeout=30)
        redirect.raise_for_status()
        candidate = redirect.headers.get("location")
        if not candidate:
            continue
        probe = requests.get(candidate, headers={"Range": "bytes=0-0"}, timeout=30)
        if probe.status_code == 206:
            signed_url = candidate
            total = int(probe.headers["content-range"].split("/")[-1])
            print(f"CDN {urlparse(candidate).netloc} accepted on attempt {attempt + 1}")
            break
    if signed_url is None:
        raise RuntimeError(f"No working signed URL for {repo_id}/{filename}")
    if destination.exists() and destination.stat().st_size == total:
        return destination
    with destination.open("wb") as handle:
        for number, start in enumerate(range(0, total, chunk_size), start=1):
            end = min(start + chunk_size - 1, total - 1)
            part = requests.get(signed_url, headers={"Range": f"bytes={start}-{end}"}, timeout=(30, 300))
            if part.status_code != 206:
                raise RuntimeError(f"Range {start}-{end} returned {part.status_code}")
            handle.write(part.content)
            if number % 10 == 0 or end == total - 1:
                print(f"downloaded {end + 1:,}/{total:,} bytes", flush=True)
    return destination


text_model_dir = WORKDIR / "text_model"
text_model_dir.mkdir(parents=True, exist_ok=True)
for filename in ["config.json", "tokenizer_config.json", "tokenizer.json", "special_tokens_map.json"]:
    response = requests.get(f"https://huggingface.co/{TEXT_MODEL_ID}/resolve/main/{filename}?download=true", timeout=30)
    response.raise_for_status()
    (text_model_dir / filename).write_bytes(response.content)
download_hf_large_file(TEXT_MODEL_ID, "model.safetensors", text_model_dir / "model.safetensors")

text_windows = build_text_windows(words, TEXT_WINDOW_SIZES, TEXT_STRIDE_DIVISOR)
tokenizer = AutoTokenizer.from_pretrained(text_model_dir)
text_model = AutoModelForSequenceClassification.from_pretrained(text_model_dir).to("cuda").eval()

scores = []
batch_size = 8
for offset in range(0, len(text_windows), batch_size):
    batch_text = text_windows.iloc[offset:offset + batch_size]["text"].tolist()
    inputs = tokenizer(batch_text, padding=True, truncation=True, max_length=1024, return_tensors="pt")
    inputs = {name: value.to("cuda") for name, value in inputs.items()}
    with torch.inference_mode():
        logits = text_model(**inputs).logits.float()
    if logits.shape[-1] == 1:
        batch_scores = torch.sigmoid(logits[:, 0])
    else:
        # The model's research usage defines class index 1 as machine-generated.
        batch_scores = torch.softmax(logits, dim=-1)[:, 1]
    scores.extend(batch_scores.cpu().tolist())

text_windows["ai_text_score"] = scores
text_windows["text_rank"] = text_windows["ai_text_score"].rank(method="first", ascending=False).astype(int)
text_windows["selected_for_audio"] = text_windows["ai_text_score"] >= TEXT_ROUTE_THRESHOLD
text_windows.sort_values("text_rank").to_csv(WORKDIR / "transcript_windows.csv", index=False)
display(text_windows.sort_values("text_rank").head(20)[
    ["text_rank", "window_size", "start_s", "end_s", "ai_text_score", "selected_for_audio", "text"]
])
print(f"Selected {int(text_windows.selected_for_audio.sum()):,}/{len(text_windows):,} transcript windows")

del text_model, tokenizer, inputs, logits
gc.collect()
torch.cuda.empty_cache()

## Step 4 — Merge detected transcript spans and slice their audio

Every transcript window above the routing threshold contributes its timestamp span. Overlapping spans are merged, padded, and completely covered with overlapping 4-second audio slices. No uniform or random audio is added.

In [ ]:
selected_windows = text_windows[text_windows.selected_for_audio].sort_values("start_s").copy()
if selected_windows.empty:
    raise RuntimeError("No transcript region passed TEXT_ROUTE_THRESHOLD; lower it and rerun Steps 3–4.")

merged = []
for row in selected_windows.itertuples():
    start_s = max(0.0, float(row.start_s) - TEXT_REGION_PADDING_SECONDS)
    end_s = float(row.end_s) + TEXT_REGION_PADDING_SECONDS
    if merged and start_s <= merged[-1]["end_s"]:
        merged[-1]["end_s"] = max(merged[-1]["end_s"], end_s)
        merged[-1]["ai_text_score"] = max(merged[-1]["ai_text_score"], float(row.ai_text_score))
        merged[-1]["window_count"] += 1
    else:
        merged.append({
            "start_s": start_s,
            "end_s": end_s,
            "ai_text_score": float(row.ai_text_score),
            "window_count": 1,
        })

detected_text_regions = pd.DataFrame(merged)
detected_text_regions.insert(0, "region_id", range(len(detected_text_regions)))
detected_text_regions.to_csv(WORKDIR / "detected_text_regions.csv", index=False)

candidates = []
for region in detected_text_regions.itertuples():
    latest_start = max(region.start_s, region.end_s - AUDIO_CLIP_SECONDS)
    starts = np.arange(region.start_s, latest_start + 1e-9, AUDIO_STRIDE_SECONDS).tolist()
    if not starts or latest_start - starts[-1] > 1e-6:
        starts.append(latest_start)
    for start_s in starts:
        candidates.append({
            "source": "transcript_detected",
            "text_region_id": int(region.region_id),
            "start_s": float(start_s),
            "end_s": float(start_s + AUDIO_CLIP_SECONDS),
            "ai_text_score": float(region.ai_text_score),
        })

candidate_clips = pd.DataFrame(candidates).drop_duplicates(subset=["start_s"]).sort_values("start_s").reset_index(drop=True)
display(detected_text_regions)
display(candidate_clips.head(30))
print(f"{len(detected_text_regions)} detected text regions -> {len(candidate_clips)} audio clips")

## Step 5 — Score clips with an Arena anti-deepfake model

The Arena-hosted Wav2Vec2 Small AntiDeepfake NDA model consumes 4 seconds of 16 kHz waveform. Class 0 is fake and class 1 is real. `spoof_score` is its softmax fake score, not a calibrated probability.

In [ ]:
import librosa
import onnxruntime as ort

audio_model_dir = WORKDIR / "audio_model"
audio_model_dir.mkdir(parents=True, exist_ok=True)
onnx_name = "wav2vec2-small-antideepfake-nda.onnx"
onnx_path = download_hf_large_file(AUDIO_MODEL_ID, onnx_name, audio_model_dir / onnx_name)

waveform, loaded_rate = librosa.load(AUDIO_PATH, sr=SAMPLE_RATE, mono=True)
assert loaded_rate == SAMPLE_RATE
audio_session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
input_meta = audio_session.get_inputs()[0]
input_name = input_meta.name
required_samples = int(input_meta.shape[1])
print("ONNX input:", input_name, input_meta.shape)


def extract_fixed_clip(start_s):
    start_sample = max(0, int(round(start_s * SAMPLE_RATE)))
    clip = waveform[start_sample:start_sample + required_samples]
    if len(clip) == 0:
        raise ValueError(f"No audio available at {start_s:.2f}s")
    if len(clip) < required_samples:
        repeats = int(np.ceil(required_samples / len(clip)))
        clip = np.tile(clip, repeats)[:required_samples]
    return clip.astype(np.float32, copy=False)


spoof_scores = []
for offset in range(0, len(candidate_clips), AUDIO_BATCH_SIZE):
    rows = candidate_clips.iloc[offset:offset + AUDIO_BATCH_SIZE]
    batch = np.stack([extract_fixed_clip(start_s) for start_s in rows.start_s])
    logits = np.asarray(audio_session.run(None, {input_name: batch})[0], dtype=np.float64)
    probabilities = np.exp(logits - logits.max(axis=1, keepdims=True))
    probabilities /= probabilities.sum(axis=1, keepdims=True)
    spoof_scores.extend(probabilities[:, 0].tolist())
    print(f"Scored {min(offset + len(rows), len(candidate_clips)):,}/{len(candidate_clips):,} clips", end="\r")
print()

candidate_clips["end_s"] = candidate_clips["start_s"] + required_samples / SAMPLE_RATE
candidate_clips["spoof_score"] = spoof_scores
candidate_clips["model_label"] = np.where(candidate_clips["spoof_score"] >= 0.5, "spoof", "bonafide")
candidate_clips["above_provisional_threshold"] = (
    candidate_clips["spoof_score"] >= PROVISIONAL_AUDIO_THRESHOLD
)
candidate_clips.sort_values("spoof_score", ascending=False).to_csv(WORKDIR / "clip_scores.csv", index=False)

display(candidate_clips.sort_values("spoof_score", ascending=False))

del audio_session, waveform
gc.collect()

## Checks — Aggregate without overclaiming

The final 0–100 video ranking is the mean of the five highest voice-model scores among transcript-selected clips. It is useful for ordering experiments, but is not a calibrated probability or production blocking threshold.

In [ ]:
sorted_audio_scores = sorted(candidate_clips["spoof_score"].tolist(), reverse=True)
top_count = min(5, len(sorted_audio_scores))
audio_top_mean = float(np.mean(sorted_audio_scores[:top_count]))
video_rank_score = 100.0 * audio_top_mean
evidence_count = int(candidate_clips["above_provisional_threshold"].sum())

summary = {
    "transcript_source": transcript_record["source"],
    "text_model": TEXT_MODEL_ID,
    "audio_model": AUDIO_MODEL_ID,
    "duration_minutes": round(words[-1]["end_s"] / 60, 3),
    "timestamped_words": len(words),
    "text_windows": len(text_windows),
    "text_route_threshold": TEXT_ROUTE_THRESHOLD,
    "selected_text_windows": int(text_windows.selected_for_audio.sum()),
    "detected_text_regions": len(detected_text_regions),
    "transcript_detected_clips_tested": len(candidate_clips),
    "max_ai_text_score": float(text_windows.ai_text_score.max()),
    "max_spoof_score": float(candidate_clips.spoof_score.max()),
    "mean_top_5_spoof_score": audio_top_mean,
    "video_rank_score_0_to_100": video_rank_score,
    "provisional_audio_threshold": PROVISIONAL_AUDIO_THRESHOLD,
    "clips_above_provisional_threshold": evidence_count,
    "provisional_multi_clip_evidence": evidence_count >= 2,
    "warning": "Experimental, uncalibrated evidence. Do not use as an automatic block decision.",
}
(WORKDIR / "run_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
display(pd.Series(summary, name="value").to_frame())

## Inspect and export

Listen to the highest-scoring clips before interpreting the result. Music, overlapping speakers, silence, compression, and ASR errors are useful failure cases to record.

In [ ]:
from IPython.display import Audio, display

# Reload only for listening; this does not run either model.
listen_waveform, _ = librosa.load(AUDIO_PATH, sr=SAMPLE_RATE, mono=True)
for row in candidate_clips.nlargest(min(5, len(candidate_clips)), "spoof_score").itertuples():
    start = int(row.start_s * SAMPLE_RATE)
    end = start + AUDIO_CLIP_SAMPLES
    print(f"{row.start_s:.2f}s | {row.source} | spoof={row.spoof_score:.3f} | text={row.ai_text_score:.3f}")
    display(Audio(listen_waveform[start:end], rate=SAMPLE_RATE))

print("Artifacts:")
for artifact in ["transcript.json", "transcript_windows.csv", "detected_text_regions.csv", "clip_scores.csv", "run_summary.json"]:
    print(WORKDIR / artifact)

# Optional download:
# from google.colab import files
# for artifact in ["transcript.json", "transcript_windows.csv", "detected_text_regions.csv", "clip_scores.csv", "run_summary.json"]:
#     files.download(str(WORKDIR / artifact))

## Next steps

Evaluate several text thresholds on labeled videos and measure two stages separately: whether the transcript router includes each known AI-voice interval, and whether the audio detector recognizes the included interval. Record end-to-end precision, recall, AUROC, and false blocks per 100 hours. Repeat after YouTube-like AAC/Opus re-encoding before calibrating a production blocking rule.